# Piso 3ºB · Prueba de voces (la primera, con audios numerados)
Misma configuración de la primera prueba (5 personajes × 5 emociones, mismas descripciones y frases), pero **cada audio lleva un número grande, el personaje, la emoción y su semilla**. Así me dices: «el 07 y el 18».

La primera vez no se guardó la semilla, así que aquellos audios exactos no se pueden recuperar; estos son nuevos con la misma receta. Por eso hay **2 tiradas** de cada combinación (cambia `TIRADAS` si quieres más o menos).

*Entorno de ejecución → Cambiar tipo → GPU T4*, y *Ejecutar todo*.

In [ ]:
!pip install -q -U qwen-tts soundfile

In [ ]:
import torch, soundfile as sf, os, random, numpy as np
from IPython.display import Audio, display, HTML
from qwen_tts import Qwen3TTSModel
model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0", dtype=torch.float16, attn_implementation="sdpa")
def seed(n):
    random.seed(n); np.random.seed(n); torch.manual_seed(n); torch.cuda.manual_seed_all(n)
os.makedirs("muestras", exist_ok=True)

In [ ]:
TIRADAS = 2   # tiradas por personaje y emoción (2 x 25 = 50 audios)

# --- misma configuración de la primera prueba ---
BASE = "Acento de España peninsular (castellano de Madrid), español natural y coloquial, nada de acento latinoamericano. "
VOCES = {
 "yo":     BASE + "Hombre de unos 22 años, voz de narrador cercano, irónico y cansado, tono medio.",
 "amigo":  BASE + "Hombre joven de unos 22 años, voz relajada y bromista, algo grave, coloquial.",
 "amiga":  BASE + "Mujer joven de unos 22 años, voz clara y enérgica, lista y sarcástica.",
 "jefe":   BASE + "Hombre de unos 55 años, voz ronca y autoritaria, casero gruñón.",
 "abuela": BASE + "Mujer de unos 75 años, voz temblorosa y cascada, vecina cotilla pero cariñosa.",
}
EMOCIONES = {
 "neutro":   "Habla con naturalidad.",
 "cansado":  "Habla muy cansado, con voz lenta y apagada, suspirando.",
 "enfadado": "Habla enfadado, con voz fuerte, tensa y rápida.",
 "alegre":   "Habla alegre y emocionado, con energía y sonriendo.",
 "panico":   "Habla en pánico, muy rápido y agudo, casi gritando.",
}
FRASES = {
 "yo":     "Son las ocho y ocho, y ya llego tarde otra vez.",
 "amigo":  "Tranquilo, tío, que sobra tiempo. ¿Quién ha dejado la sartén en el suelo?",
 "amiga":  "No pasa nada, es puntualidad. Pero la nevera ha hecho un ruido muy raro.",
 "jefe":   "Esto es un piso, no un hotel. Quiero el alquiler antes del lunes.",
 "abuela": "Ay, hijo, qué ruido hacéis. En mis tiempos esto no pasaba.",
}

In [ ]:
# Generar y mostrar: cada audio con su NÚMERO grande
n = 0; tabla = []
for who, desc in VOCES.items():
    for emo, ins in EMOCIONES.items():
        for t in range(TIRADAS):
            n += 1; sd = 1000 + n; seed(sd)
            wavs, sr = model.generate_voice_design(text=FRASES[who], language="Spanish", instruct=desc + " " + ins)
            f = f"muestras/{n:02d}_{who}_{emo}_semilla{sd}.wav"; sf.write(f, wavs[0], sr)
            tabla.append((n, who, emo, sd))
            display(HTML(f"<div style='font:700 28px sans-serif;margin-top:14px'>Nº {n:02d} &nbsp;·&nbsp; {who} &nbsp;·&nbsp; {emo}"
                         f"<span style='font:14px sans-serif;color:#888'> &nbsp;semilla {sd}</span></div>"))
            display(Audio(f))

In [ ]:
# Lista para consultar y descarga de todo
for r in tabla: print(f"Nº {r[0]:02d}  {r[1]:7s} {r[2]:9s} semilla {r[3]}")
!zip -qr muestras.zip muestras
from google.colab import files; files.download("muestras.zip")

## Qué me tienes que decir
Los **números** de los audios que te convencieron (por ejemplo «07, 18 y 33») y, si puedes, qué te gustó: el acento, la emoción o la voz.